<a href="https://colab.research.google.com/github/4cekay/IAT460-A2-RuleBasedSystems/blob/kitchen/final_system.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Concept: Hybrid System** (Option 3)

*"Visual Haiku"*

Concept:

Part 1: Poem Creation
- Generative grammar set:
  - structure
  - vocabulary
  - thematic groups
  - syllabic groups

Part 2: Visualizer
- L-System:
  - mapping character groups into instructions
  - drawing


# Setup

In [2]:
!pip install ColabTurtle

  Preparing metadata (setup.py) ... done
  Created wheel for ColabTurtle: filename=colabturtle-2.1.0-py3-none-any.whl size=7674 sha256=6e69ebafd237729d1dacc317683bcd8256d10049ccea2129095e72b89a39faef
  Stored in directory: /root/.cache/pip/wheels/8d/30/a2/2b8e815bd3673383f33c0dd80d33ef7651a7c257c528515158
Successfully built ColabTurtle


In [3]:
import random
import ColabTurtle.Turtle as t
from IPython.display import clear_output

In [4]:
# Map each letter of the alphabet to colours to use in the L-system

alphabet_colours = [
    "#ebebeb", # 0: a
    "#3b1182", # 1: b
    "#25d21b", # 2: c
    "#4259d7", # 3: d
    "#ebebeb", # 4: e
    "#c258f4", # 5: f
    "#226939", # 6: g
    "#7d8988", # 7: h
    "#ebebeb", # 8: i
    "#3db1a5", # 9: j
    "#1bd280", # 10: k
    "#e6d295", # 11: l
    "#bb0f8f", # 12: m
    "#a330a8", # 13: n
    "#ebebeb", # 14: o
    "#682f9f", # 15: p
    "#7cc33f", # 16: q
    "#f1bc3b", # 17: r
    "#e62f2b", # 18: s
    "#203aca", # 19: t
    "#ebebeb", # 20: u
    "#9268d2", # 21: v
    "#f2a4eb", # 22: w
    "#981310", # 23: x
    "#ebebeb", # 24: y
    "#ed6045", # 25: z
]

# Generative Grammar System

To generate a (basic) haiku poem:
- 3 lines (stanzas)
- Stanza syllable pattern of: 5-7-5
# #
Additional rules/guidelines for simplicity:
- Singular nouns ONLY
- Present-tense verbs ONLY
- No weighted options, but purposeful word choices provided
- Parts of speech grouped by syllable count for easy tracking

In [5]:
# A Dictionary defining basic Haiku poem structure

haiku_grammar = {
    # Root of syntactic structure
    'HAIKU': [['SYLL_FIVE', 'SYLL_SEVEN', 'SYLL_FIVE']],

    # Possible stanza types per word count
    'SYLL_FIVE': [
        ['FIVE_5W'],
        ['FIVE_4W'],
        ['FIVE_3W'],
        ['FIVE_2W'],
        ['FIVE_1W']
    ],
    'SYLL_SEVEN': [
        ['SEVEN_7W'],
        ['SEVEN_6W'],
        ['SEVEN_5W'],
        ['SEVEN_4W'],
        ['SEVEN_3W'],
        ['SEVEN_2W']
    ],

    # Possible structure per stanza type
    'FIVE_4W': [
        ['A_S1', 'N_S1', 'ADV_S2', 'V_S1'],
        ['ADJ_S2', 'ADJ_S1', 'C_S1', 'ADJ_S1'],
        #['ADV', 'V', 'N', 'V'],
        #['N', 'V', 'C', 'V']
    ],

    'SEVEN_5W': [
        ['A_S1', 'N_S1', 'V_S2', 'C_S1', 'V_S2'],
    ],
    # Parts of speech per syllable count:

    # Articles
    'A_S1': ['the', 'a'],

    # Conjunctions
    'C_S1': ['and', 'yet', 'but', 'or'],

    # Prepositions
    'P_S1': ['to', 'from', 'that', 'as', 'on'],
    'P_S2': ['under', 'over', 'against', 'after', 'before'],

    # Nouns
    'N_S1': ['bird', 'star', 'girl', 'world', 'sky'],

    # Adjectives
    'ADJ_S1': ['sweet', 'kind', 'bright', 'sad', 'quick', 'loud', 'dark', 'warm', 'hot', 'cold', 'wet'],
    'ADJ_S2': ['blissful', 'peaceful', 'pretty', 'happy', 'somber', 'hopeful', 'careful', 'sleepy', 'angry', 'joyful'],

    # Verbs
    'V_S1': ['sings', 'glows', 'smiles', 'says', 'looks', 'cries', 'wilts', 'floats', 'breathes', 'sighs'],  # present tense
    'V_S2': ['shining', 'beaming', 'searching', 'dancing', 'crying', 'wishing', 'hoping', 'dreaming', 'writing', 'saying'], # present

    # Adverbs
    'ADV_S2': ['brightly', 'coolly', 'darkly', 'loudly', 'warmly'],

}

In [8]:
def generate_haiku(symbol='HAIKU', grammar=haiku_grammar):
    """
    Generate a haiku using words from the above grammar system.
    """

    # In-progress poem results (after every recursive step)
    current_result = {
        'poem': "",
        'lines_syllables': [],
        'word_count': [],
        'word_syllables': [],
        'part_of_speech': [],
        'colours': []
    }

    if symbol in grammar:

    # First: track NON-TERMINAL symbols as they're chosen in each recursive step
    # by adding them to the saved collections above

      # Symbol is a chosen LINE:
      if symbol in ['SYLL_FIVE', 'SYLL_SEVEN']:
        current_result['lines_syllables'].append(symbol)

      # Symbol is a chosen LINE STRUCTURE:
      elif symbol.startswith(("FIVE_", "SEVEN_")):
        current_result['word_count'].append(symbol[len(symbol)-2])

      # Symbol is a chosen WORD TYPE (Part of Speech + Syll Count):
      elif symbol[len(symbol)-2] == "S":
        current_result['word_syllables'].append(symbol[len(symbol)-1])
        current_result['part_of_speech'].append(symbol.split('_')[0])


    # Select a random value from the symbol's line of rules
      production = random.choice(grammar[symbol])

      if isinstance(production, list): # non-terminals, parent node
        words = [] # temporarily hold all collected word terminals

        # Move recursively down the rules tree
        for sym in production:
          sub_result = generate_haiku(sym, grammar) # child node, until terminal

          # Traversing back up: update the current node's result collection
          # with the child node's values

          words.append(sub_result['poem'])
          current_result['lines_syllables'].extend(sub_result['lines_syllables'])
          current_result['word_count'].extend(sub_result['word_count'])
          current_result['word_syllables'].extend(sub_result['word_syllables'])
          current_result['colours'].extend(sub_result['colours'])
          current_result['part_of_speech'].extend(sub_result['part_of_speech'])


        # Line Level (Semi-Final): all words in one line + newline
        if symbol in ['SYLL_FIVE', 'SYLL_SEVEN']:
          current_result['poem'] = ' '.join(words).strip() + '\n'

        # Poem Level (Final): all comprising lines
        elif symbol == 'HAIKU':
          current_result['poem'] = ''.join(words)

        # Word Level: each part of speech
        else:
          current_result['poem'] = ' '.join(words).strip()

      elif isinstance(production, str): # terminals, save as chosen word before UP
        current_result['poem'] = production
        current_result['colours'] = [ord(production[0]) - 97] # colour map index

    return current_result


In [92]:
new_haiku_1 = generate_haiku()

new_haiku_1

{'poem': 'the world loudly glows\nthe world writing or wishing\nangry loud but kind\n',
 'lines_syllables': ['SYLL_FIVE', 'SYLL_SEVEN', 'SYLL_FIVE'],
 'word_count': ['4', '5', '4'],
 'word_syllables': ['1',
  '1',
  '2',
  '1',
  '1',
  '1',
  '2',
  '1',
  '2',
  '2',
  '1',
  '1',
  '1'],
 'part_of_speech': ['A',
  'N',
  'ADV',
  'V',
  'A',
  'N',
  'V',
  'C',
  'V',
  'ADJ',
  'ADJ',
  'C',
  'ADJ'],
 'colours': [19, 22, 11, 6, 19, 22, 22, 14, 22, 0, 11, 1, 10]}

In [93]:
print(new_haiku_1['poem'])

the world loudly glows
the world writing or wishing
angry loud but kind



# L-System

Visualizing the resulting haiku using Turtle Graphics:
- Map the poem's pieces into a string of L-System instructions to draw a unique fractal tree:
  - Each letter character is a different colour
  - Syllable count of a word become the number of iterations on that word's branch
  - Part of speech determines which side of the tree the branch protrudes from

Steps:
- Start with the trunk
- SAVE
- Turn one way depending on part of speech
- COLOUR CHANGE
- Draw branch in that direction depending on ha



IDEA:
- straight line of "tree trunk" in the middle, achieved with a saved state of the turtle that will be restored after each branch
- each branch represents a different word in the poem
  - total number of child branches depends on the **WORD COUNT**
  - the initial turn (left or right, meaning the branch either protrudes from the top side or bottom side of the horizontal branch we see) is mapped to the **PART OF SPEECH**
  - number of iterations on that word branch depends on the **WORD SYLLABLE COUNT**
  - colour of each sub-branch depends on the **FIRST LETTER** of the word

## Helper Functions

1. Base L-System creation code (from lab)

Instructions Key:
- **D** = Go forward, drawing
- **M** = Go forward, moving only
- **R** = Turn right
- **L** = Turn left
- **[** = Save current state
- **]** = Restore previous state

In [26]:
def create_l_system(iterations, axiom, rules):
    """Generate L-System instructions based on axiom and rules."""
    result = axiom
    for _ in range(iterations):
        new_string = ""
        for char in result:
            new_string += rules.get(char, char)
        result = new_string
    return result

2. Custom helper function: Haiku fractal tree instruction compiler

In [31]:
def haiku_instructions(haiku_result, rules):
  """
  Compile data from randomly generated poem into a final L-System instructions string
  """

  final_instructions = ""

  # Derive total word count from length of dict entry with each word's syllable count
  total_words = len(haiku_result['word_syllables'])

  # Map out which side each part of speech should branch out from (Left = Up, Right = Down)
  pos_sides = {
      'N': 'L', # Nouns, Verbs, Adjectives, Adverbs branch UPWARDS
      'V': 'L',
      'ADJ': 'L',
      'ADV': 'L',
      'A': 'R', # Articles, Conjunctions, Prepositions branch DOWNWARDS
      'C': 'R',
      'P': 'R',
  }

  # Handle the poem line-by-line > word-for-word:
  current_word = 0

  # Get number of words in each stanza
  for count_str in haiku_result['word_count']:
    line_word_count = int(count_str)

    for _ in range(line_word_count):

      # Start with the base of the tree, then SAVE position
      final_instructions += 'D'
      final_instructions += '['

      # Add colour instruction based on first letter of word
      colour_index = haiku_result['colours'][current_word]
      final_instructions += f'C{colour_index}_'

      # Branch out either upwards or downwards depending on Part of Speech
      pos = haiku_result['part_of_speech'][current_word]
      final_instructions += pos_sides.get(pos)

      # Fractal branch using syllable count of word as iterations to determine length
      syllables = int(haiku_result['word_syllables'][current_word])

      branch_string = create_l_system(syllables, axiom="D", rules=rules)
      final_instructions += branch_string

      # RESTORE position back to branch start for the next word
      final_instructions += ']'

      # Move onto the next word
      current_word += 1

    final_instructions += 'N' # [N]ext line terminator

  return final_instructions

3. Drawing functions (customized from lab base code)

In [66]:
def setup_turtle():
  # Load in Turtle and make it invisible
  t.initializeTurtle()
  t.speed(13)
  t.hideturtle()

  # Starting position
  start_x = 0
  start_y = t.window_height() // 3

  t.penup()
  t.goto(start_x, start_y)
  t.pendown()
  t.setheading(0) # Horizontal-facing

  # Space in between each stanza's tree
  line_spacing = 115

  return start_x, start_y, line_spacing

In [97]:
def draw_haiku_tree(poem, instructions, angle, distance):
    """Draw the L-System using turtle graphics.

    Parameters:
    - instructions: string of L-System commands
    - angle: turning angle in degrees
    - distance: forward movement distance
    """

    # Get Turtle starting position
    x, y, line_spacing = setup_turtle()

    # Initialize the turtle drawing loop (iterate through all instructions)
    i=0 # Current instruction
    word = 0 # Current word
    stack = [] # Turtle pos/ang states

    while i < len(haiku_instructions_string):
      cmd = haiku_instructions_string[i]

      # First, extract branch colour and set it as the Turtle's active colour
      if cmd == 'C':
        end = haiku_instructions_string.find("_", i)
        word_colour = int(haiku_instructions_string[i+1 : end]) # e.g. C[19]_
        t.pencolor(alphabet_colours[word_colour])

        stack.append((t.position(), t.heading()))

        # Move onto the next instruction
        i= end+1
        continue

      # Drawing/Movement instructions
      elif cmd == 'D':
        t.forward(distance)
      elif cmd == 'M':
        t.penup()
        t.forward(distance)
        t.pendown()

      # Left and Right turn instructions
      elif cmd == 'L':
        t.left(angle)
      elif cmd == 'R':
        t.right(angle)

      # Save and Restore Turtle positions (pos + ang)
      elif cmd == '[':
            stack.append((t.position(), t.heading()))
      elif cmd == ']':  # Restore previous state
            position, heading = stack.pop()
            t.penup()
            t.goto(position)
            t.setheading(heading)
            t.pendown()

      # Newline (move onto the next stanza)
      elif cmd == 'N':
        y += line_spacing
        t.penup()
        t.goto(x, y)
        t.setheading(0)
        t.pendown()

      # Move onto the next line's instructions
      i += 1

    print(poem)

## Compiling Instructions

Once we have a generated haiku:

In [98]:
# Fetch all the poem details to use
metadata = new_haiku_1

# Rules for recursive drawing of a fractal tree branch
haiku_branch = {
    'D': 'DDDD[LLDD]DD[RRDD]'
}

# Final instruction string output:
haiku_instructions_string = haiku_instructions(metadata, haiku_branch)

In [99]:
print(haiku_instructions_string)

D[C19_RDDDD[LLDD]DD[RRDD]]D[C22_LDDDD[LLDD]DD[RRDD]]D[C11_LDDDD[LLDD]DD[RRDD]DDDD[LLDD]DD[RRDD]DDDD[LLDD]DD[RRDD]DDDD[LLDD]DD[RRDD][LLDDDD[LLDD]DD[RRDD]DDDD[LLDD]DD[RRDD]]DDDD[LLDD]DD[RRDD]DDDD[LLDD]DD[RRDD][RRDDDD[LLDD]DD[RRDD]DDDD[LLDD]DD[RRDD]]]D[C6_LDDDD[LLDD]DD[RRDD]]ND[C19_RDDDD[LLDD]DD[RRDD]]D[C22_LDDDD[LLDD]DD[RRDD]]D[C22_LDDDD[LLDD]DD[RRDD]DDDD[LLDD]DD[RRDD]DDDD[LLDD]DD[RRDD]DDDD[LLDD]DD[RRDD][LLDDDD[LLDD]DD[RRDD]DDDD[LLDD]DD[RRDD]]DDDD[LLDD]DD[RRDD]DDDD[LLDD]DD[RRDD][RRDDDD[LLDD]DD[RRDD]DDDD[LLDD]DD[RRDD]]]D[C14_RDDDD[LLDD]DD[RRDD]]D[C22_LDDDD[LLDD]DD[RRDD]DDDD[LLDD]DD[RRDD]DDDD[LLDD]DD[RRDD]DDDD[LLDD]DD[RRDD][LLDDDD[LLDD]DD[RRDD]DDDD[LLDD]DD[RRDD]]DDDD[LLDD]DD[RRDD]DDDD[LLDD]DD[RRDD][RRDDDD[LLDD]DD[RRDD]DDDD[LLDD]DD[RRDD]]]ND[C0_LDDDD[LLDD]DD[RRDD]DDDD[LLDD]DD[RRDD]DDDD[LLDD]DD[RRDD]DDDD[LLDD]DD[RRDD][LLDDDD[LLDD]DD[RRDD]DDDD[LLDD]DD[RRDD]]DDDD[LLDD]DD[RRDD]DDDD[LLDD]DD[RRDD][RRDDDD[LLDD]DD[RRDD]DDDD[LLDD]DD[RRDD]]]D[C11_LDDDD[LLDD]DD[RRDD]]D[C1_RDDDD[LLDD]DD[RRDD]]D[C10_LDD

In [106]:
draw_haiku_tree(metadata['poem'], haiku_instructions_string, 10, 15)

the world loudly glows
the world writing or wishing
angry loud but kind

